# IMO Health — Diagnosis Refinement Agent (Medication-Guided, Two-Phase)

This notebook provides a **Diagnosis Refinement Agent** that refines diagnoses to their most specific form
using a **two-phase approach**: medication-guided hierarchy drill-down followed by note-driven refinement axes.

## What Does This Agent Do?

Given a clinical note, this agent will:
1. **Extract** base diagnoses (stripped of all qualifiers) + medications + clinical qualifiers in priority order
2. **Normalize** the base diagnosis (domain=Problem) and each medication (domain=Medication) via IMO Precision Normalize API
3. **Phase 1 — Medication-Guided Drill-Down**: Iteratively traverse `domainNarrowerByMedications` in the KG, picking candidates that match documented qualifiers
4. **Phase 2 — Note-Driven Refinement Axes**: Apply `allowedRefinements` + `refinementNarrower` one at a time for remaining qualifiers
5. **Present** the refined diagnosis with ICD-10 codes and the full refinement path

| Component | Technology |
|-----------|------------|
| LLM | AWS Bedrock / OpenAI / Anthropic / Azure OpenAI (configurable) |
| Tools | IMO Normalize API + KG GraphQL (`domainNarrowerByMedications`, `allowedRefinements`, `refinementNarrower`) |
| Agent | LangGraph ReAct Agent |
| Auth | OAuth2 client_credentials grant |

## Prerequisites

- `config.json` file with IMO API + LLM credentials (copy from `config.json.template`)
- For AWS Bedrock: AWS credentials (SageMaker execution role or explicit keys)
- For OpenAI/Anthropic/Azure: respective API keys in config.json
- Python 3.10+

## Step 1: Install Dependencies

Run this cell once, then restart the kernel.

Installs support for all LLM providers. You only need credentials for the one you choose.

In [ ]:
%pip install -q --upgrade \
    "langchain-core>=1.0.0" \
    "langchain>=1.0.0" \
    "langchain-aws>=0.2.0" \
    "langchain-openai>=0.3.0" \
    "langchain-anthropic>=0.3.0" \
    "langgraph>=0.2.0" \
    boto3 botocore requests nest_asyncio python-dotenv

## Step 2: Configuration

Loads credentials from `config.json`. Create one from `config.json.template` if it doesn't exist.

### LLM Provider

Set `llm.provider` in `config.json` to one of:
- `"bedrock"` — AWS Bedrock (Claude via AWS)
- `"openai"` — OpenAI (GPT-4o, etc.)
- `"anthropic"` — Anthropic direct API (Claude)
- `"azure_openai"` — Azure OpenAI Service

In [ ]:
import os
import sys
import json
import pathlib
import nest_asyncio

nest_asyncio.apply()

# --- Ensure notebook directory is on path for local imports ---
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# --- Load config.json ---
candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json',
]
cfg_path = next((p for p in candidates if p.exists()), None)
if cfg_path is None:
    raise FileNotFoundError(
        'config.json not found. Copy config.json.template to config.json and fill in your credentials.'
    )

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

# --- IMO API credentials ---
imo_cfg = cfg.get('imo_api', {})
imo_kg_cfg = cfg.get('imo_kg', {})

os.environ['IMO_NORMALIZE_CLIENT_ID'] = imo_cfg.get('client_id', '')
os.environ['IMO_NORMALIZE_SECRET'] = imo_cfg.get('client_secret', '')
os.environ['IMO_KG_CLIENT_ID'] = imo_kg_cfg.get('client_id', '') or imo_cfg.get('client_id', '')
os.environ['IMO_KG_CLIENT_SECRET'] = imo_kg_cfg.get('client_secret', '') or imo_cfg.get('client_secret', '')

# --- LLM configuration ---
llm_cfg = cfg.get('llm', {})
LLM_PROVIDER = llm_cfg.get('provider', 'bedrock')

# --- AWS credentials from config.json (for Bedrock) ---
aws_cfg = cfg.get('aws', {})
if aws_cfg.get('access_key_id'):
    os.environ.pop('AWS_PROFILE', None)
    os.environ['AWS_ACCESS_KEY_ID'] = aws_cfg['access_key_id']
    os.environ['AWS_SECRET_ACCESS_KEY'] = aws_cfg['secret_access_key']
    os.environ['AWS_SESSION_TOKEN'] = aws_cfg.get('session_token', '')
    os.environ['AWS_DEFAULT_REGION'] = aws_cfg.get('region', 'us-east-1')

print(f'Config loaded from : {cfg_path.resolve()}')
print(f'LLM Provider       : {LLM_PROVIDER}')
print(f'Normalize URL      : {imo_cfg.get("normalize_url", "https://api.imohealth.com/precision/normalize")}')
print(f'KG GraphQL URL     : {imo_cfg.get("graphql_url", "https://api.imohealth.com/knowledgegraph/graphql/")}')

## Step 3: Initialize LLM

Creates the LLM client based on the configured provider.  
Temperature is set to 0 for deterministic outputs across all providers.

In [ ]:
def create_llm(provider: str, llm_cfg: dict):
    """Create LLM instance based on the configured provider."""

    if provider == 'bedrock':
        from langchain_aws import ChatBedrockConverse
        bedrock_cfg = llm_cfg.get('bedrock', {})
        model_id = bedrock_cfg.get('model_id', 'us.anthropic.claude-haiku-4-5-20251001-v1:0')
        region = bedrock_cfg.get('region', 'us-east-1')
        llm = ChatBedrockConverse(
            model_id=model_id,
            region_name=region,
            temperature=0,
            max_tokens=64000,
            provider="anthropic",
        )
        print(f'LLM ready: AWS Bedrock ({model_id}, {region})')
        return llm

    elif provider == 'openai':
        from langchain_openai import ChatOpenAI
        openai_cfg = llm_cfg.get('openai', {})
        model = openai_cfg.get('model', 'gpt-4o')
        llm = ChatOpenAI(
            model=model,
            api_key=openai_cfg.get('api_key', ''),
            temperature=0,
        )
        print(f'LLM ready: OpenAI ({model})')
        return llm

    elif provider == 'anthropic':
        from langchain_anthropic import ChatAnthropic
        anthropic_cfg = llm_cfg.get('anthropic', {})
        model = anthropic_cfg.get('model', 'claude-sonnet-4-20250514')
        llm = ChatAnthropic(
            model=model,
            api_key=anthropic_cfg.get('api_key', ''),
            temperature=0,
            max_tokens=64000,
        )
        print(f'LLM ready: Anthropic ({model})')
        return llm

    elif provider == 'azure_openai':
        from langchain_openai import AzureChatOpenAI
        azure_cfg = llm_cfg.get('azure_openai', {})
        llm = AzureChatOpenAI(
            azure_deployment=azure_cfg.get('deployment', ''),
            azure_endpoint=azure_cfg.get('endpoint', ''),
            api_key=azure_cfg.get('api_key', ''),
            api_version=azure_cfg.get('api_version', '2024-02-15-preview'),
            temperature=0,
        )
        print(f'LLM ready: Azure OpenAI ({azure_cfg.get("deployment", "")})')
        return llm

    else:
        raise ValueError(
            f'Unknown LLM provider: "{provider}". '
            f'Supported: bedrock, openai, anthropic, azure_openai'
        )


llm = create_llm(LLM_PROVIDER, llm_cfg)

## Step 4: Define Agent Tools

Nine tools power the two-phase refinement workflow:

| Tool | Phase | Purpose |
|------|-------|---------|
| `normalize_medical_term` | Setup | Normalize diagnosis (domain=Problem) or medications (domain=Medication) via IMO Precision Normalize API |
| `get_lexical` | Both | Query KG for a concept's core data — refinements, mappings, hierarchy, relationships |
| `get_domain_hierarchy` | Both | Get narrower (children) or broader (parents) concepts in the domain hierarchy |
| `get_treatments_for_problem` | Both | Get associated, supportive, and contraindicated treatments for a problem |
| `get_caused_problems` | Both | Get complications caused by a condition or adverse effects of a medication |
| `get_diagnostics_for_problem` | Both | Get diagnostic tests and lab procedures relevant to a problem |
| `get_medication_diagnosis_proto` | Phase 1 | Query KG `domainNarrowerByMedications` — returns narrower problem concepts linked to supplied medications |
| `get_allowed_refinements` | Phase 2 | Get refinement groups (type, laterality, severity, chronicity, etc.) available for a concept |
| `get_narrower_sequential_refinements` | Phase 2 | Apply refinements one at a time via `refinementNarrower` to resolve the most specific concept |

In [ ]:
import json
from langchain_core.tools import tool
from kg_api_client import KGApiClient

_kg_client = KGApiClient()


@tool
def normalize_medical_term(input_term: str, domain: str = "Problem") -> str:
    """Normalize a medical term using IMO Precision Normalize API.

    Use domain="Problem" for diagnoses/conditions.
    Use domain="Medication" for medications/drugs.

    Returns the normalized concept with:
    - title: canonical name
    - default_lexical_code: the stable IMO identifier used for KG lookups
    - score: confidence score
    - icd10_codes: mapped ICD-10-CM codes (for Problem domain)
    """
    result = _kg_client.normalize_medical_term(input_term, domain)
    return json.dumps(result, indent=2)


@tool
def get_lexical(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Query the IMO Knowledge Graph for a concept's core data.
    - For medication (domain="medication"): treatedProblems, contraindicatedProblems, causedProblems
    - For problem (domain="problem"): allowedRefinements, narrower subtypes, refinementFamilies, mappings

    NOTE: For problem treatments/contraindications, use get_treatments_for_problem instead.

    Args:
        imo_lexical_code: The default_lexical_code from normalize results. REQUIRED.
        domain: "problem" or "medication"
    """
    result = _kg_client.get_lexical(imo_lexical_code, domain)
    return json.dumps(result, indent=2)


@tool
def get_domain_hierarchy(imo_lexical_code: str, direction: str = "narrower", domain: str = "Problem") -> str:
    """Get domain hierarchy (children or parents) for a problem concept.
    - direction="narrower": Get subtypes/children (e.g., heart failure -> systolic HF, diastolic HF)
    - direction="broader": Get parent concepts (e.g., systolic HF -> heart failure)

    Also returns synonyms for the concept.

    Args:
        imo_lexical_code: The IMO lexical code. REQUIRED.
        direction: "narrower" for children/subtypes, "broader" for parent concepts
        domain: Use "Problem"
    """
    result = _kg_client.get_domain_hierarchy(imo_lexical_code, direction, domain)
    return json.dumps(result, indent=2)


@tool
def get_treatments_for_problem(imo_lexical_code: str) -> str:
    """Get treatment-related medications for a problem/diagnosis from the KG.
    Returns: associatedTreatments, contraindicatedTreatments, supportiveTreatments.

    Use this when you need to find what medications treat, prevent, or are contraindicated for a condition.

    Args:
        imo_lexical_code: The default_lexical_code of the problem (from normalize with domain="Problem")
    """
    result = _kg_client.get_treatments_for_problem(imo_lexical_code)
    return json.dumps(result, indent=2)


@tool
def get_caused_problems(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get problems/complications caused by a condition or medication.
    - For problem: returns causedProblems (complications) and causativeAgents (medications that cause it)
    - For medication: returns causedProblems (adverse effects)

    Args:
        imo_lexical_code: The default_lexical_code from normalize results
        domain: "problem" or "medication"
    """
    result = _kg_client.get_caused_problems(imo_lexical_code, domain)
    return json.dumps(result, indent=2)


@tool
def get_diagnostics_for_problem(imo_lexical_code: str) -> str:
    """Get diagnostic tests and lab procedures relevant to a problem/diagnosis.
    Returns: associatedProcedures.

    Args:
        imo_lexical_code: The default_lexical_code of the problem (from normalize with domain="Problem")
    """
    result = _kg_client.get_diagnostics_for_problem(imo_lexical_code)
    return json.dumps(result, indent=2)


@tool
def get_medication_diagnosis_proto(diagnosis_code: str, medication_codes: list[str]) -> str:
    """Query the IMO Knowledge Graph for narrower diagnosis concepts linked to medications.

    Uses the `domainNarrowerByMedications` field on ProblemLexical to find
    child concepts of the base diagnosis that have therapeutic ties to the
    supplied medications.

    Results are sorted so non-leaf concepts (numberOfDomainChildren > 0) appear
    first, ordered by descending children count.

    Args:
        diagnosis_code: The base diagnosis lexical code (from normalize with domain="Problem").
        medication_codes: List of medication lexical codes (from normalize with domain="Medication").

    Returns:
        Dict with the base diagnosis title and its domainNarrowerByMedications list.
        Each entry has: code, title, numberOfDomainChildren.
        - numberOfDomainChildren > 0 means the concept can be drilled further
        - numberOfDomainChildren == 0 means it is a leaf (most specific)

    Use this tool iteratively: pick the best candidate matching clinical context,
    then call again with that candidate's code to drill deeper, until a leaf is reached.
    """
    result = _kg_client.get_medication_diagnosis_proto(diagnosis_code, medication_codes)

    if result.get("success") and result.get("lexical"):
        items = result["lexical"].get("domainNarrowerByMedications", [])
        if items:
            items.sort(key=lambda x: -(x.get("numberOfDomainChildren") or 0))
            result["lexical"]["domainNarrowerByMedications"] = items

    return json.dumps(result, indent=2)


@tool
def get_allowed_refinements(imo_lexical_code: str, domain: str = "Problem") -> str:
    """Get allowed refinements for a problem concept. Returns refinement groups
    (type, laterality, severity, chronicity, etc.) that can further narrow a diagnosis.

    Each refinement has a code, title, and group (e.g., group.title = "Laterality").
    Use the refinement code with get_narrower_sequential_refinements to apply it.

    Args:
        imo_lexical_code: The IMO lexical code of the problem
        domain: Use "Problem".
    """
    result = _kg_client.get_allowed_refinements(imo_lexical_code, domain)
    return json.dumps(result, indent=2)


@tool
def get_narrower_sequential_refinements(
    imo_lexical_code: str,
    refinement_sequence: list[list[str]],
    include_mappings: bool = True,
    domain: str = "Problem",
) -> str:
    """Apply refinements sequentially to resolve the most specific concept.

    Takes a base concept and applies one or more refinement steps. Each step
    narrows the concept further. The response includes the resolved concept(s)
    with their ICD-10 mappings and remaining allowedRefinements.

    Args:
        imo_lexical_code: Base concept lexical code
        refinement_sequence: Array of arrays, each inner array is one refinement step.
            Example: [["230"], ["362636588"]] applies two refinements sequentially.
        include_mappings: Include ICD-10 mappings in response
        domain: Use "Problem".
    """
    result = _kg_client.get_narrower_sequential_refinements(
        imo_lexical_code, refinement_sequence, include_mappings, domain
    )
    return json.dumps(result, indent=2)


tools = [
    normalize_medical_term,
    get_lexical,
    get_domain_hierarchy,
    get_treatments_for_problem,
    get_caused_problems,
    get_diagnostics_for_problem,
    get_allowed_refinements,
    get_narrower_sequential_refinements,
    get_medication_diagnosis_proto,
]
print(f'Tools ready ({len(tools)}): {[t.name for t in tools]}')

## Step 5: System Prompt & Create Agent

The agent follows a **two-phase** refinement workflow:

### Phase 1 — Medication-Guided Hierarchy Drill-Down
1. **Extract** base diagnoses (stripped of ALL qualifiers) + medications + clinical qualifiers in priority order
2. **Normalize** base diagnosis and each medication
3. **Iteratively call** `get_medication_diagnosis_proto` — pick the candidate matching the highest-priority documented qualifier, recurse until leaf or no match

### Phase 2 — Note-Driven Refinement Axes
4. Starting from Phase 1's result, apply **one refinement at a time** using `get_allowed_refinements` + `get_narrower_sequential_refinements`
5. **Present** the refined diagnosis with ICD-10, full refinement path, and reasoning

### Explicit Documentation Rule
The agent may ONLY pick a candidate/refinement if justified by:
- (A) Explicitly written in the note by the provider
- (B) Directly observable from the medication list (e.g., "without insulin" if no insulin prescribed)
- (C) Type/category identification from medication class (e.g., Metformin → type 2 DM)
- (D) Validated clinical assessment tools (e.g., PHQ-9, EF%, NYHA class)

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a medication-based diagnosis refinement agent. You refine diagnoses to their most specific form using a two-phase approach: medication-guided hierarchy drill-down followed by note-driven refinement axes.

## WORKFLOW

### Step 1: Extract (NO TOOL CALLS)
From the clinical note, extract:
- Base diagnoses (strip ALL qualifiers — type, severity, laterality, chronicity, complications)
  Example: "chronic systolic congestive heart failure" → base = "heart failure"
  Example: "type 2 diabetes mellitus with nephropathy" → base = "diabetes mellitus"
- Medications associated with each diagnosis
- Clinical qualifiers from the note in PRIORITY ORDER:
  1. Type/category (e.g., type 2, primary, systolic)
  2. Chronicity (e.g., chronic, acute, acute on chronic)
  3. Complications/manifestations (e.g., nephropathy, retinopathy, congestive)
  4. Severity/stage (e.g., moderate, stage 3, NYHA class 3)
  5. Laterality (e.g., left, right, bilateral)
  6. Other qualifiers (e.g., uncontrolled, with insulin use)

Present a brief summary of extraction, then immediately proceed to Step 2.

### Step 2: Normalize Base Diagnosis and Medications
1. Call normalize_medical_term(input_term=<base diagnosis>, domain="Problem") to get its default_lexical_code
2. For each medication, call normalize_medical_term(input_term=<med>, domain="Medication") to get its default_lexical_code
Collect all medication codes into a list.

### Step 3: PHASE 1 — Medication-Guided Hierarchy Drill-Down
Call get_medication_diagnosis_proto(diagnosis_code=<base dx code>, medication_codes=[<all med codes>]).
This returns narrower concepts of the base diagnosis that have ties to those medications.

**Iterative drill-down rules:**
1. Each returned candidate has `code`, `title`, and `numberOfDomainChildren`
2. Pick the candidate matching the HIGHEST-PRIORITY unused qualifier from Step 1 (type > chronicity > complication > severity > laterality) — BUT ONLY if that qualifier is EXPLICITLY DOCUMENTED in the note text (see EXPLICIT DOCUMENTATION RULE below)
3. Among qualifying candidates, prefer those with numberOfDomainChildren > 0 (non-leaves) over leaves
4. Apply ONE qualifier per depth level — do not jump to a concept matching 3 qualifiers at once when a single-qualifier match with children exists
5. After picking, call get_medication_diagnosis_proto AGAIN with the picked candidate's code and the SAME medication_codes list
6. STOP when:
   - numberOfDomainChildren = 0 (leaf reached), OR
   - The tool returns an empty list (no further med-linked children), OR
   - NO candidate at this level matches an EXPLICITLY DOCUMENTED qualifier (STOP HERE — proceed to Phase 2), OR
   - 8 rounds completed (safety bound)

**If Phase 1 returns EMPTY on the very first call** (no med-linked subtypes at all):
- Skip to Step 4 (Phase 2) directly — the medications don't provide hierarchy guidance for this condition, so use refinement axes instead.

### Step 4: PHASE 2 — Note-Driven Refinement Axes (ONE REFINEMENT AT A TIME)
Starting from the concept reached at the end of Phase 1 (or the base diagnosis if Phase 1 was empty):

Apply refinements ONE AT A TIME iteratively:

1. Call get_allowed_refinements on the CURRENT concept's code
2. Check which UNUSED qualifiers from Step 1 match available refinement groups. Also check medication-observable qualifiers (rule B) like "without long term use of insulin"
3. Pick the HIGHEST-PRIORITY matching refinement code
4. Call get_narrower_sequential_refinements with ONLY that single refinement: refinement_sequence=[["<code>"]]
5. If it returns a resolved concept:
   - That concept becomes your NEW current concept
   - Go back to step 1 and repeat (check allowed refinements on the new concept for remaining qualifiers)
6. If it returns EMPTY:
   - Skip that refinement and try the next matching one
   - If all remaining refinements return empty, STOP — use the current concept as final
7. STOP when no more matching refinements remain or 5 iterations completed

### Step 5: Present Results
Use normalize_medical_term on the final refined diagnosis (domain="Problem") to get ICD-10 codes, then present:

#### Medication-Based Diagnosis Refinement Results

| Original Diagnosis | Medication | Refined Diagnosis | ICD-10 | Reasoning |
|---|---|---|---|---|
| (base) | (drug) | (refined) | (code) | (why) |

For each refined diagnosis include:
- **IMO Lexical Code:** (code)
- **IMO KG Link:** https://studio.imohealth.com/#/terminology-browser/graph?id=(code)
- **Refinement Path:** Show full drill-down path:
  - Phase 1 steps: base → picked concept (reason) → picked concept (reason) → ... → leaf
  - Phase 2 steps: leaf → +refinement applied → final concept
- **Phase 1 Depth:** Number of drill-down levels used
- **Phase 2 Refinements Applied:** List of refinement axes used, or "None needed"

## RULES
- Always use default_lexical_code from normalize results for KG lookups
- NEVER call normalize with an empty string — skip any term that is empty or unclear
- Do NOT wait for user input after extraction — refine all diagnoses automatically
- NEVER fabricate ICD-10 codes — get them from normalize tool
- NEVER invent or assume a more specific diagnosis if the KG did not return data for it. If a tool returns empty/no results, use the last successfully resolved concept and clearly state "No further KG data available for this refinement"
- Do NOT use your own medical knowledge to pick a more specific code — only use what the KG tools return
- Before each tool call, briefly explain why you're calling it
- When Phase 1 returns many candidates (50+), focus on candidates matching your highest-priority unused qualifier — do not attempt to list or evaluate all of them
- CRITICAL — EXPLICIT DOCUMENTATION RULE: During Phase 1 drill-down AND Phase 2 refinement selection, you may ONLY pick a candidate/refinement whose qualifier is justified by one of the following:
  (A) EXPLICITLY WRITTEN in the note by the provider — the exact word or direct synonym appears in Assessment, HPI, or Problem List text, OR
  (B) DIRECTLY OBSERVABLE from the medication list — factual medication-profile qualifiers:
      - "without long-term current use of insulin" → valid if no insulin in med list
      - "treated with insulin" → valid if insulin IS in the med list
      - "treated with oral medication" → valid if oral diabetes meds are prescribed
  (C) TYPE/CATEGORY identification from medication class — the medication's pharmacological class directly identifies the disease subtype (e.g., Metformin is exclusively a type 2 DM drug → "type 2 diabetes mellitus" is valid). This applies ONLY to type/category distinctions, NOT to severity or control status.
  (D) VALIDATED CLINICAL ASSESSMENT TOOLS — when a disease-specific standardized assessment is documented in the note, it identifies the disease subtype:
      - PHQ-9 score documented → identifies "major depressive disorder" (PHQ-9 is specific to MDD screening)
      - Ejection fraction (EF%) → identifies HFrEF vs HFpEF (EF ≤40% = reduced, ≥50% = preserved)
      - NYHA class documented → identifies functional classification
      - FEV1/FVC ratio → identifies obstructive vs restrictive lung disease
      This applies to TYPE/CATEGORY identification only, NOT to severity or control status.
  - What is NOT allowed (requires clinical inference/judgment):
    - Control status: "uncontrolled", "well controlled", "poorly controlled" — NEVER infer from labs or clinical context, must be explicitly documented by the provider
    - HbA1c values do NOT justify any control-status qualifier
    - "Continue current regimen" does NOT justify "well controlled" or "controlled"
    - PHQ-9 score does NOT justify severity level (mild/moderate/severe) unless the provider explicitly documents the severity word
  - If NO candidate at a given level has a qualifier satisfying (A), (B), (C), or (D), STOP PHASE 1 at the current level (even if non-leaf candidates exist) and proceed to Phase 2 with the current concept
  - This rule takes ABSOLUTE PRIORITY over "prefer non-leaves" — a non-leaf candidate with an inferred control-status qualifier must be REJECTED in favor of stopping"""

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print('Diagnosis Refinement Agent ready (Two-Phase: Medication-Guided + Note-Driven Refinements).')

## Step 6: Clinical Note

The following clinical note will be used to test the two-phase refinement agent.
It contains a base diagnosis with multiple qualifiers and medications that guide both phases.

In [ ]:
CLINICAL_NOTE = """Chief Complaint: Follow-up for diabetes management.

HPI: 58-year-old male with a history of type 2 diabetes mellitus diagnosed 12 years ago, presenting for routine follow-up. Patient reports increased thirst and frequent urination over the past 2 months. He has been compliant with his current oral medication regimen. Denies hypoglycemic episodes. Reports tingling and numbness in bilateral feet for the past 6 months, worse at night.

PMH:
- Type 2 diabetes mellitus with diabetic peripheral neuropathy
- Hypertension
- Hyperlipidemia
- Chronic kidney disease, stage 3a (eGFR 52)

Current Medications:
- Metformin 1000 mg BID
- Glipizide 10 mg daily
- Lisinopril 20 mg daily
- Atorvastatin 40 mg daily

Allergies: NKDA.

Vitals: BP 138/82, HR 76, T 98.4°F, BMI 31.2.

Exam:
- Bilateral feet: decreased monofilament sensation, absent ankle reflexes
- No skin ulcers or deformities
- Fundoscopic exam: no retinopathy noted

Labs:
- HbA1c: 8.2%
- Fasting glucose: 186 mg/dL
- Serum creatinine: 1.4 mg/dL
- eGFR: 52 mL/min/1.73m2
- Urine microalbumin/creatinine ratio: 45 mg/g (mildly elevated)

Assessment:
1) Type 2 diabetes mellitus with diabetic peripheral neuropathy. Currently on oral agents only (no insulin). Peripheral neuropathy confirmed on exam.
2) Chronic kidney disease stage 3a — likely diabetic nephropathy given microalbuminuria and longstanding DM.

Plan:
- Continue Metformin 1000 mg BID (renal function acceptable)
- Continue Glipizide 10 mg daily
- Refer to podiatry for neuropathy management
- Repeat HbA1c in 3 months
- Recheck renal function and microalbumin in 6 months

Base Diagnosis:
- Diabetes mellitus

Medications:
- Metformin
- Glipizide

Please refine this diagnosis to its most specific form."""

print("Clinical note loaded (Type 2 DM with neuropathy + Metformin/Glipizide).")
print(f"Length: {len(CLINICAL_NOTE)} characters")

## Step 7: Run the Agent

Send the clinical note to the agent and observe the two-phase refinement process:
- **Phase 1**: Medication-guided drill-down via `domainNarrowerByMedications`
- **Phase 2**: Note-driven refinements via `allowedRefinements` + `refinementNarrower`

In [ ]:
import asyncio
from IPython.display import display, HTML, Markdown
import html as html_module


class AgentUI:
    """Rich HTML display for agent streaming output."""

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #57039E 0%, #7C0EDD 50%, #57039E 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">Diagnosis Refinement Agent (Two-Phase)</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Phase 1: Medication-Guided Drill-Down &rarr; Phase 2: Note-Driven Refinement Axes</p>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:12px;">Type <b>quit</b> to end · <b>reset</b> to clear history · <b>Kernel Interrupt</b> to stop mid-generation</p>
        </div>
        """))

    @staticmethod
    def status(message):
        display(HTML(f'<div style="color:#5f6368; font-size:12px; font-style:italic; padding:4px 0;">{html_module.escape(message)}</div>'))

    @staticmethod
    def reasoning(text):
        clean = html_module.escape(text.strip())
        display(HTML(f"""
        <div style="background:#fff8e1; border-left:4px solid #f59e0b; border-radius:0 8px 8px 0; padding:10px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#d97706;">&#128161; Agent Reasoning</b>
            <div style="margin-top:6px; color:#92400e; white-space:pre-wrap; max-height:200px; overflow-y:auto;">{clean}</div>
        </div>
        """))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2, default=str))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">&#128295; Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:400])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">&#9989; Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 400 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))

    @staticmethod
    def stopped():
        display(HTML("""
        <div style="background:#fce8e6; border:1px solid #c5221f; border-radius:8px; padding:12px 16px; margin:8px 0; text-align:center;">
            <b style="color:#c5221f;">Session ended.</b>
        </div>
        """))


async def run_inference(clinical_note: str):
    """Run the diagnosis refinement agent on a clinical note."""
    ui = AgentUI()
    ui.header()

    messages = [{'role': 'user', 'content': clinical_note}]
    final_content = ""
    last_reasoning = ""

    async for chunk in agent.astream(
        {'messages': messages},
        stream_mode='updates'
    ):
        for node_name, node_output in chunk.items():
            if node_name == 'agent':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    # text content FIRST - capture reasoning before tool calls
                    if hasattr(msg, 'content') and msg.content:
                        if isinstance(msg.content, str) and msg.content:
                            final_content = msg.content
                            last_reasoning = msg.content
                        elif isinstance(msg.content, list):
                            text_parts = []
                            for block in msg.content:
                                if isinstance(block, dict) and block.get('type') == 'text':
                                    text_parts.append(block['text'])
                                elif isinstance(block, str):
                                    text_parts.append(block)
                            if text_parts:
                                joined = ''.join(text_parts)
                                final_content = joined
                                last_reasoning = joined

                    # tool calls SECOND - show reasoning then tool call
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        if last_reasoning:
                            ui.reasoning(last_reasoning)
                            last_reasoning = ""
                        for tc in msg.tool_calls:
                            ui.tool_call(tc['name'], tc.get('args', {}))

            elif node_name == 'tools':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    if hasattr(msg, 'content'):
                        ui.tool_result(
                            getattr(msg, 'name', 'tool'),
                            msg.content
                        )

    if final_content:
        ui.separator()
        ui.agent_response(final_content)

    return final_content


result = await run_inference(CLINICAL_NOTE)


## Step 8: Interactive Chat Mode

Use this cell for multi-turn interaction with the agent.  
Paste any clinical note with medications and the agent will refine the diagnoses using both phases.

Commands: `quit` to end, `reset` to clear history.

In [ ]:
import asyncio
import json
import html as html_module
from IPython.display import display, HTML, Markdown


class AgentUI:

    @staticmethod
    def header():
        display(HTML("""
        <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #57039E 0%, #7C0EDD 50%, #57039E 100%);
                    border-radius:12px; color:white; margin-bottom:16px;">
            <h2 style="margin:0;">Diagnosis Refinement Agent (Two-Phase)</h2>
            <p style="margin:4px 0 0 0; opacity:0.9;">Phase 1: Medication-Guided Drill-Down &rarr; Phase 2: Note-Driven Refinement Axes</p>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:12px;">Type <b>quit</b> to end · <b>reset</b> to clear history · <b>Kernel Interrupt</b> to stop mid-generation</p>
        </div>
        """))

    @staticmethod
    def status(message):
        display(HTML(f'<div style="color:#5f6368; font-size:12px; font-style:italic; padding:4px 0;">{html_module.escape(message)}</div>'))

    @staticmethod
    def reasoning(text):
        clean = html_module.escape(text.strip())
        display(HTML(f"""
        <div style="background:#fff8e1; border-left:4px solid #f59e0b; border-radius:0 8px 8px 0; padding:10px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#d97706;">&#128161; Agent Reasoning</b>
            <div style="margin-top:6px; color:#92400e; white-space:pre-wrap; max-height:200px; overflow-y:auto;">{clean}</div>
        </div>
        """))

    @staticmethod
    def tool_call(tool_name, parameters):
        params_json = html_module.escape(json.dumps(parameters, indent=2, default=str))
        display(HTML(f"""
        <div style="background:#e8f0fe; border:1px solid #1a73e8; border-radius:8px; padding:12px 16px; margin:8px 0; font-family:monospace; font-size:13px;">
            <b style="color:#1a73e8;">&#128295; Tool Call: {html_module.escape(tool_name)}</b>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:12px; overflow-x:auto;">{params_json}</pre>
        </div>
        """))

    @staticmethod
    def tool_result(tool_name, result):
        result_str = str(result)
        preview = html_module.escape(result_str[:400])
        full = html_module.escape(result_str)
        char_count = len(result_str)
        display(HTML(f"""
        <div style="background:#e6f4ea; border:1px solid #137333; border-radius:8px; padding:12px 16px; margin:8px 0; font-size:13px;">
            <b style="color:#137333;">&#9989; Result: {html_module.escape(tool_name)}</b>
            <details style="margin-top:8px;">
                <summary style="cursor:pointer; font-weight:600; font-size:12px; color:#137333;">View full result ({char_count:,} chars)</summary>
                <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:8px; font-size:11px; max-height:300px; overflow:auto;">{full}</pre>
            </details>
            <pre style="background:rgba(0,0,0,0.04); padding:8px; border-radius:4px; margin-top:4px; font-size:11px; opacity:0.7;">{preview}{'...' if char_count > 400 else ''}</pre>
        </div>
        """))

    @staticmethod
    def agent_response(text):
        display(Markdown(text))

    @staticmethod
    def separator():
        display(HTML('<hr style="border:none; border-top:2px solid #dadce0; margin:16px 0;">'))

    @staticmethod
    def stopped():
        display(HTML("""
        <div style="background:#fce8e6; border:1px solid #c5221f; border-radius:8px; padding:12px 16px; margin:8px 0; text-align:center;">
            <b style="color:#c5221f;">Session ended.</b>
        </div>
        """))


ui = AgentUI()
chat_messages = []
ui.header()

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() == 'quit':
        ui.stopped()
        break
    if user_input.lower() == 'reset':
        chat_messages = []
        ui.status('Conversation reset. Paste a new clinical note.')
        continue

    chat_messages.append({'role': 'user', 'content': user_input})
    ui.status('Agent is thinking...')

    final_content = ""
    last_reasoning = ""

    async for chunk in agent.astream(
        {'messages': chat_messages},
        stream_mode='updates'
    ):
        for node_name, node_output in chunk.items():
            if node_name == 'agent':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    if hasattr(msg, 'content') and msg.content:
                        if isinstance(msg.content, str) and msg.content:
                            final_content = msg.content
                            last_reasoning = msg.content
                        elif isinstance(msg.content, list):
                            text_parts = []
                            for block in msg.content:
                                if isinstance(block, dict) and block.get('type') == 'text':
                                    text_parts.append(block['text'])
                                elif isinstance(block, str):
                                    text_parts.append(block)
                            if text_parts:
                                joined = ''.join(text_parts)
                                final_content = joined
                                last_reasoning = joined

                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        if last_reasoning:
                            ui.reasoning(last_reasoning)
                            last_reasoning = ""
                        for tc in msg.tool_calls:
                            ui.tool_call(tc['name'], tc.get('args', {}))

            elif node_name == 'tools':
                msgs = node_output.get('messages', [])
                for msg in msgs:
                    if hasattr(msg, 'content'):
                        ui.tool_result(
                            getattr(msg, 'name', 'tool'),
                            msg.content
                        )

    if final_content:
        ui.separator()
        ui.agent_response(final_content)
        chat_messages.append({'role': 'assistant', 'content': final_content})

    ui.separator()
